# Database Logging with Ibis - PostgreSQL and DuckDB

In this notebook, we will demonstrate how to set up database logging for the SQL AI Agent using the `get_ibis_connection()` utility. We will:
- Create a helper function to convert JSON log files to CSV for DuckDB
- Initialize PostgreSQL log table using `get_ibis_connection()`
- Register log table with DuckDB using CSV conversion
- Generate sample logs and query them from both databases

This approach provides a unified workflow for both databases using the same connection utility.

## Setup and Imports

Let's start by importing the required libraries and modules:

In [1]:
import sys
import os
import socket

# Local WSL/conda helper: course examples sometimes use Docker's hostname
# "postgres". In this local notebook, redirect that hostname to localhost.
if not hasattr(socket, "_sql_agent_original_getaddrinfo"):
    socket._sql_agent_original_getaddrinfo = socket.getaddrinfo

    def _sql_agent_local_postgres_getaddrinfo(host, *args, **kwargs):
        if host == "postgres":
            host = "127.0.0.1"
        return socket._sql_agent_original_getaddrinfo(host, *args, **kwargs)

    socket.getaddrinfo = _sql_agent_local_postgres_getaddrinfo

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import pandas as pd
import json
import logging
from datetime import datetime
import ibis
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

# SQL AI Agent imports
from sql_ai_agent.data import get_ibis_connection
from sql_ai_agent.log_database import (
    init_postgres_log_schema,
    init_duckdb_log_schema,
    DatabaseLogHandler,
    verify_log_table
)
from sql_ai_agent.logger import SQLAgentLogger

print("✅ All imports successful!")

✅ All imports successful!


## Helper Function: Convert JSON Logs to CSV

Since DuckDB works well with CSV files through `get_ibis_connection()`, we need a function to convert JSON log files to CSV format:

In [3]:
def json_logs_to_csv(json_file_path: str, csv_file_path: str) -> pd.DataFrame:
    """
    Convert JSON log file to CSV format for DuckDB ingestion.
    
    Args:
        json_file_path: Path to JSON log file (one JSON object per line)
        csv_file_path: Path to output CSV file
        
    Returns:
        DataFrame with processed log data
    """
    # Read JSON log file (newline-delimited JSON)
    logs = []
    with open(json_file_path, 'r') as f:
        for line in f:
            try:
                log_entry = json.loads(line.strip())
                logs.append(log_entry)
            except json.JSONDecodeError:
                continue  # Skip invalid lines
    
    if not logs:
        raise ValueError("No valid JSON log entries found")
    
    # Convert to DataFrame
    df = pd.DataFrame(logs)
    
    # Ensure all required columns exist
    required_columns = [
        'timestamp', 'level', 'logger', 'message', 
        'session_id', 'operation_type'
    ]
    
    for col in required_columns:
        if col not in df.columns:
            df[col] = None
    
    # Handle extra fields - convert dict/list columns to JSON strings
    extra_fields_data = []
    for idx, row in df.iterrows():
        extra_dict = {}
        for col in df.columns:
            if col not in required_columns + ['exception', 'extra_fields']:
                if pd.notna(row[col]):
                    extra_dict[col] = row[col]
        
        # Also include existing extra_fields if present
        if 'extra_fields' in row and isinstance(row['extra_fields'], dict):
            extra_dict.update(row['extra_fields'])
        
        extra_fields_data.append(json.dumps(extra_dict) if extra_dict else None)
    
    # Create final DataFrame with schema matching database
    result_df = pd.DataFrame({
        'timestamp': pd.to_datetime(df['timestamp']),
        'level': df['level'],
        'logger': df['logger'],
        'message': df['message'],
        'session_id': df['session_id'],
        'operation_type': df['operation_type'],
        'extra_fields': extra_fields_data,
        'exception': df.get('exception', None)
    })
    
    # Save to CSV
    result_df.to_csv(csv_file_path, index=False)
    print(f"✅ Converted {len(result_df)} log entries to CSV: {csv_file_path}")
    
    return result_df

print("✅ Helper function defined")

✅ Helper function defined


## PostgreSQL Workflow

In this section, we will:
1. Connect to PostgreSQL using `get_ibis_connection()`
2. Initialize the log schema
3. Set up database logging
4. Generate sample logs
5. Query the logs

### Step 1: Configure PostgreSQL Connection

In [4]:
# PostgreSQL configuration
# Values are loaded from ../.env when available. Defaults target local Postgres.
postgres_config = {
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "password"),
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "database": os.getenv("POSTGRES_DATABASE", "my_db"),
}

if postgres_config["host"] == "postgres":
    raise ValueError(
        "POSTGRES_HOST is set to 'postgres', which only works inside Docker. "
        "For local WSL/conda notebooks, set POSTGRES_HOST=localhost in ../.env."
    )

# Table name for logs
log_table_name = "sql_agent_logs"

print(
    "✅ PostgreSQL configuration set: "
    f"{postgres_config['host']}:{postgres_config['port']}/{postgres_config['database']}"
)

✅ PostgreSQL configuration set: localhost:5432/my_db


### Step 2: Connect to PostgreSQL

**Note:** For this workflow, we'll use direct `ibis.postgres.connect()` since `get_ibis_connection()` is designed for loading data tables, not for administrative tasks like creating log schemas.

In [5]:
# Connect to PostgreSQL
con_postgres = ibis.postgres.connect(**postgres_config)

print(f"✅ Connected to PostgreSQL: {postgres_config['host']}:{postgres_config['port']}/{postgres_config['database']}")

✅ Connected to PostgreSQL: localhost:5432/my_db


### Step 3: Initialize PostgreSQL Log Schema

In [6]:
# Initialize log table with optimized schema
init_postgres_log_schema(
    con=con_postgres,
    table_name=log_table_name,
    schema="public"
)

print(f"\n✅ PostgreSQL log table initialized!")
print(f"   Table: public.{log_table_name}")
print(f"   Indexes: 6 created for query performance")

✅ PostgreSQL log table '"public"."sql_agent_logs"' initialized successfully
   - Table created with optimized schema
   - 6 indexes created for query performance
   - Ready to receive log entries

✅ PostgreSQL log table initialized!
   Table: public.sql_agent_logs
   Indexes: 6 created for query performance


### Step 4: Verify Table Creation

In [7]:
# Verify table exists
stats = verify_log_table(con_postgres, log_table_name, "postgres")

print("📊 Table Statistics:")
print(f"   Table exists: {stats['exists']}")
print(f"   Total logs: {stats['row_count']}")

📊 Table Statistics:
   Table exists: True
   Total logs: 6


### Step 5: Set Up Database Logging Handler

In [8]:
# Create logger for PostgreSQL
pg_logger = logging.getLogger('sql_ai_agent_postgres')
pg_logger.setLevel(logging.INFO)
pg_logger.handlers.clear()

# Add database handler
pg_db_handler = DatabaseLogHandler(
    con=con_postgres,
    table_name=log_table_name,
    db_type="postgres",
    schema="public",
    level=logging.INFO
)
pg_logger.addHandler(pg_db_handler)

# Add console handler for visibility
console_handler = logging.StreamHandler()
console_handler.setFormatter(
    logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
)
pg_logger.addHandler(console_handler)

print("✅ PostgreSQL database logging configured!")

✅ PostgreSQL database logging configured!


### Step 6: Generate Sample Logs to PostgreSQL

In [9]:
# Create SQLAgentLogger adapter
pg_agent_logger = SQLAgentLogger(
    pg_logger,
    extra={'session_id': 'notebook-postgres-ibis'}
)

# Write sample logs
pg_agent_logger.info(
    "PostgreSQL logging initialized via ibis connection",
    extra={
        'operation_type': 'initialization',
        'database_type': 'postgres',
        'connection_method': 'ibis.postgres.connect'
    }
)

pg_agent_logger.info(
    "Sample query executed",
    extra={
        'operation_type': 'query_result',
        'query': 'SELECT COUNT(*) FROM employees',
        'rows_returned': 100,
        'success': True
    }
)

pg_agent_logger.info(
    "LLM invocation completed",
    extra={
        'operation_type': 'llm_invocation',
        'model_name': 'gpt-4o',
        'estimated_prompt_tokens': 1250,
        'agent_config': {
            'model': 'gpt-4o',
            'read_only': True,
            'memory_enabled': True
        }
    }
)

print("\n✅ Sample logs written to PostgreSQL!")

2026-05-24 00:16:43,404 - INFO - PostgreSQL logging initialized via ibis connection
2026-05-24 00:16:43,422 - INFO - Sample query executed
2026-05-24 00:16:43,425 - INFO - LLM invocation completed



✅ Sample logs written to PostgreSQL!


### Step 7: Query PostgreSQL Logs

In [10]:
# Query recent logs
query = f"""
SELECT 
    id,
    timestamp,
    level,
    message,
    operation_type,
    session_id
FROM {log_table_name}
ORDER BY timestamp DESC
LIMIT 10;
"""

recent_logs = con_postgres.sql(query).execute()

print("📋 Recent logs from PostgreSQL:")
print(recent_logs.to_string(index=False))

📋 Recent logs from PostgreSQL:
 id                        timestamp level                                            message operation_type             session_id
  9 2026-05-24 00:16:43.425386+00:00  INFO                           LLM invocation completed llm_invocation notebook-postgres-ibis
  8 2026-05-24 00:16:43.422314+00:00  INFO                              Sample query executed   query_result notebook-postgres-ibis
  7 2026-05-24 00:16:43.404397+00:00  INFO PostgreSQL logging initialized via ibis connection initialization notebook-postgres-ibis
  6 2026-05-23 23:13:21.152868+00:00  INFO                           LLM invocation completed llm_invocation notebook-postgres-ibis
  5 2026-05-23 23:13:21.149775+00:00  INFO                              Sample query executed   query_result notebook-postgres-ibis
  4 2026-05-23 23:13:21.146621+00:00  INFO PostgreSQL logging initialized via ibis connection initialization notebook-postgres-ibis
  3 2026-05-23 23:12:26.609910+00:00  INFO   

In [11]:
# Export current logs to file for DuckDB demonstration
export_query = f"""
SELECT 
    timestamp,
    level,
    logger,
    message,
    session_id,
    operation_type,
    extra_fields::text as extra_fields,
    exception
FROM {log_table_name}
ORDER BY timestamp DESC;
"""

logs_df = con_postgres.sql(export_query).execute()

# Save to CSV for DuckDB
os.makedirs('../logs', exist_ok=True)
csv_path = '../logs/agent_logs.csv'
logs_df.to_csv(csv_path, index=False)

print(f"✅ Exported {len(logs_df)} logs to CSV: {csv_path}")

✅ Exported 9 logs to CSV: ../logs/agent_logs.csv


## DuckDB Workflow

In this section, we will:
1. Use `get_ibis_connection()` to load the CSV logs into DuckDB
2. Initialize the DuckDB log schema (if needed)
3. Query the logs from DuckDB
4. Demonstrate live logging to DuckDB

### Step 1: Load Logs into DuckDB using get_ibis_connection()

In [12]:
# Use get_ibis_connection to load CSV into DuckDB
con_duckdb = get_ibis_connection(
    backend="duckdb",
    tbl_name=log_table_name,
    duckdb_csv_path=csv_path
)

print(f"✅ Loaded logs into DuckDB table: {log_table_name}")
print(f"   Connection type: DuckDB (in-memory)")

✅ Loaded logs into DuckDB table: sql_agent_logs
   Connection type: DuckDB (in-memory)


### Step 2: Verify DuckDB Table

In [13]:
# Check table schema
query = f"DESCRIBE {log_table_name};"
schema_info = con_duckdb.con.execute(query).df()

print("📊 DuckDB Table Schema:")
print(schema_info.to_string(index=False))

📊 DuckDB Table Schema:
   column_name column_type null  key default extra
     timestamp     VARCHAR  YES None    None  None
         level     VARCHAR  YES None    None  None
        logger     VARCHAR  YES None    None  None
       message     VARCHAR  YES None    None  None
    session_id     VARCHAR  YES None    None  None
operation_type     VARCHAR  YES None    None  None
  extra_fields     VARCHAR  YES None    None  None
     exception      DOUBLE  YES None    None  None


In [14]:
# Count rows
query = f"SELECT COUNT(*) as count FROM {log_table_name};"
count_result = con_duckdb.con.execute(query).df()

print(f"\n📊 Total logs in DuckDB: {count_result['count'][0]}")


📊 Total logs in DuckDB: 9


### Step 3: Query DuckDB Logs

In [15]:
# Query recent logs
query = f"""
SELECT 
    timestamp,
    level,
    message,
    operation_type,
    session_id
FROM {log_table_name}
ORDER BY timestamp DESC
LIMIT 10;
"""

recent_logs_duckdb = con_duckdb.con.execute(query).df()

print("📋 Recent logs from DuckDB:")
print(recent_logs_duckdb.to_string(index=False))

📋 Recent logs from DuckDB:
                       timestamp level                                            message operation_type             session_id
2026-05-24 00:16:43.425386+00:00  INFO                           LLM invocation completed llm_invocation notebook-postgres-ibis
2026-05-24 00:16:43.422314+00:00  INFO                              Sample query executed   query_result notebook-postgres-ibis
2026-05-24 00:16:43.404397+00:00  INFO PostgreSQL logging initialized via ibis connection initialization notebook-postgres-ibis
2026-05-23 23:13:21.152868+00:00  INFO                           LLM invocation completed llm_invocation notebook-postgres-ibis
2026-05-23 23:13:21.149775+00:00  INFO                              Sample query executed   query_result notebook-postgres-ibis
2026-05-23 23:13:21.146621+00:00  INFO PostgreSQL logging initialized via ibis connection initialization notebook-postgres-ibis
2026-05-23 23:12:26.609910+00:00  INFO                           LLM invocati

### Step 4: Query Logs by Operation Type

In [16]:
# Count logs by operation type
query = f"""
SELECT 
    operation_type,
    COUNT(*) as count
FROM {log_table_name}
WHERE operation_type IS NOT NULL
GROUP BY operation_type
ORDER BY count DESC;
"""

operation_stats = con_duckdb.con.execute(query).df()

print("📊 Logs by Operation Type (DuckDB):")
print(operation_stats.to_string(index=False))

📊 Logs by Operation Type (DuckDB):
operation_type  count
initialization      3
llm_invocation      3
  query_result      3


### Step 5: Extract JSON Fields from extra_fields

In [17]:
# Query with JSON extraction (if extra_fields contains JSON)
query = f"""
SELECT 
    timestamp,
    level,
    message,
    operation_type,
    extra_fields
FROM {log_table_name}
WHERE extra_fields IS NOT NULL
    AND extra_fields != ''
    AND extra_fields != '{{}}'
ORDER BY timestamp DESC
LIMIT 5;
"""

detailed_logs = con_duckdb.con.execute(query).df()

print("📊 Detailed Logs with Extra Fields (DuckDB):")
print(detailed_logs.to_string(index=False))

📊 Detailed Logs with Extra Fields (DuckDB):
                       timestamp level                                            message operation_type                                                                                                                                                                                                          extra_fields
2026-05-24 00:16:43.425386+00:00  INFO                           LLM invocation completed llm_invocation {"model_name": "gpt-4o", "session_id": "notebook-postgres-ibis", "agent_config": {"model": "gpt-4o", "read_only": true, "memory_enabled": true}, "operation_type": "llm_invocation", "estimated_prompt_tokens": 1250}
2026-05-24 00:16:43.422314+00:00  INFO                              Sample query executed   query_result                                                          {"query": "SELECT COUNT(*) FROM employees", "success": true, "session_id": "notebook-postgres-ibis", "rows_returned": 100, "operation_type": "query_result"}

### Step 6: (Optional) Set Up Live Logging to DuckDB

For live logging, we can create a separate DuckDB connection with a file-based database:

In [18]:
# Create a persistent DuckDB database for live logging
os.makedirs('../logs', exist_ok=True)
duckdb_file = '../logs/agent_logs.duckdb'

con_duckdb_persistent = ibis.duckdb.connect(duckdb_file)

# Initialize log schema
init_duckdb_log_schema(
    con=con_duckdb_persistent,
    table_name=log_table_name
)

print(f"✅ Persistent DuckDB database created: {duckdb_file}")
print(f"   Table: {log_table_name}")

✅ DuckDB log table 'sql_agent_logs' initialized successfully
   - Table created with optimized schema
   - Auto-increment sequence created
   - Ready to receive log entries
✅ Persistent DuckDB database created: ../logs/agent_logs.duckdb
   Table: sql_agent_logs


In [19]:
# Set up database logging handler for DuckDB
duckdb_logger = logging.getLogger('sql_ai_agent_duckdb')
duckdb_logger.setLevel(logging.INFO)
duckdb_logger.handlers.clear()

# Add database handler
duckdb_db_handler = DatabaseLogHandler(
    con=con_duckdb_persistent,
    table_name=log_table_name,
    db_type="duckdb",
    level=logging.INFO
)
duckdb_logger.addHandler(duckdb_db_handler)

# Add console handler
duckdb_console_handler = logging.StreamHandler()
duckdb_console_handler.setFormatter(
    logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
)
duckdb_logger.addHandler(duckdb_console_handler)

print("✅ DuckDB live logging configured!")

✅ DuckDB live logging configured!


In [20]:
# Write sample logs to DuckDB
duckdb_agent_logger = SQLAgentLogger(
    duckdb_logger,
    extra={'session_id': 'notebook-duckdb-ibis'}
)

duckdb_agent_logger.info(
    "DuckDB logging initialized via ibis connection",
    extra={
        'operation_type': 'initialization',
        'database_type': 'duckdb',
        'storage': 'persistent_file'
    }
)

duckdb_agent_logger.info(
    "Sample analytical query",
    extra={
        'operation_type': 'query_result',
        'query': 'SELECT department, AVG(salary) FROM employees GROUP BY department',
        'rows_returned': 5,
        'success': True
    }
)

print("\n✅ Sample logs written to DuckDB!")

--- Logging error ---
Traceback (most recent call last):
  File "/home/farjam/sql-agent-ai/sql_ai_agent/log_database.py", line 498, in emit
    self._insert_duckdb(
  File "/home/farjam/sql-agent-ai/sql_ai_agent/log_database.py", line 596, in _insert_duckdb
    self.con.con.execute(insert_sql)
_duckdb.ConstraintException: Constraint Error: Duplicate key "id: 1" violates primary key constraint.
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/farjam/miniconda3/envs/sql-agent-ai/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/farjam/miniconda3/envs/sql-agent-ai/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/home/farjam/miniconda3/envs/sql-agent-ai/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/farjam/mini


✅ Sample logs written to DuckDB!


In [21]:
# Query logs from persistent DuckDB
query = f"""
SELECT 
    id,
    timestamp,
    level,
    message,
    operation_type
FROM {log_table_name}
ORDER BY timestamp DESC
LIMIT 5;
"""

duckdb_logs = con_duckdb_persistent.con.execute(query).df()

print("📋 Recent logs from persistent DuckDB:")
print(duckdb_logs.to_string(index=False))

📋 Recent logs from persistent DuckDB:
 id                  timestamp level                                        message operation_type
  2 2026-05-23 23:13:21.779500  INFO                        Sample analytical query   query_result
  1 2026-05-23 23:13:21.757360  INFO DuckDB logging initialized via ibis connection initialization


## Summary

In this notebook, we demonstrated:

### ✅ PostgreSQL Workflow
- Connected to PostgreSQL using `ibis.postgres.connect()`
- Initialized log schema with optimized indexes
- Set up database logging handler
- Generated sample logs
- Queried logs using SQL
- Exported logs to CSV for DuckDB

### ✅ DuckDB Workflow
- Used `get_ibis_connection()` to load CSV logs into DuckDB
- Queried logs from in-memory DuckDB table
- Created persistent DuckDB database for live logging
- Demonstrated log querying with JSON field extraction

### ✅ Key Features
- **Unified approach**: Same log schema for both databases
- **CSV conversion**: Easy data transfer between PostgreSQL and DuckDB
- **Flexible storage**: In-memory DuckDB for analysis, persistent for logging
- **SQL queries**: Standard SQL works across both databases

## Next Steps

1. **Integrate with SqlAgent**
   - Use these logging setups in your SQL AI Agent
   - Choose PostgreSQL for production, DuckDB for development

2. **Build Analytics**
   - Create dashboards from log data
   - Track token usage and costs
   - Monitor error rates

3. **Production Deployment**
   - Use PostgreSQL for shared logging
   - Set up log rotation and archival
   - Configure monitoring alerts

## Resources

- 📘 [DATABASE_LOGGING.md](../docs/DATABASE_LOGGING.md) - Complete documentation
- 📄 [database_logging_examples.py](../examples/database_logging_examples.py) - Python examples
- 🔧 [log_database.py](../sql_ai_agent/log_database.py) - Source code
- 📓 [database_logging_setup.ipynb](../examples/database_logging_setup.ipynb) - Alternative setup notebook

---

**Happy Logging! 🎉**